# Music Annotation Project
## Notebook 66 v2.1 - Minimal Human Semantic Review of 59R3 Retained Regions

**Runtime:** CPU. In Colab select **Runtime -> Run all**.

Only two human labels are required per region:

1. **What is this region?** `ANNOTATION`, `PRINT`, `MIXED`, `UNCERTAIN`, or `MASK_ERROR`.
2. **If annotation is present, what kind?** `PALE_PENCIL`, `DARK_WRITING`, `PART_CATALOGUE`, `LONG_MARK_ARROW`, or `OTHER`. For non-annotation decisions the notebook stores `NOT_APPLICABLE` automatically.

Machine-derived cluster, family, probability and morphology evidence is retained automatically but hidden until the human judgement is first saved. The initial queue is page-diversified and capped at approximately 40 regions per sampling family, about 200 reviews in total.

This is evidence collection only. It does not itself make reviewed regions training truth, freeze a challenger, or authorise production use.

**v2.1 fix:** the queue adapter reads the actual Notebook 59R3 `flag_*_candidate` columns and fails if any requested family resolves to zero candidates.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gradio pandas numpy opencv-python-headless pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, zipfile, platform, sys
import cv2, gradio as gr, numpy as np, pandas as pd

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_59R3_RUN_OVERRIDE=ROOT / 'notebook59R3_retained_morphology' / '20261114_113203_notebook59R3_retained_morphology'
OUTPUT_ROOT=ROOT/'notebook66_v2_minimal_semantic_review'
RUN=OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook66_v2')
TABLES=RUN/'tables'; REVIEW=RUN/'review'; AUDIT=RUN/'audit'; UPLOAD=RUN/'upload'
for p in [RUN,TABLES,REVIEW,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
SEED=42; INITIAL_PER_FAMILY=40; MAX_INITIAL_REVIEWS=200; CONTEXT_MARGIN=110; DISPLAY_HEIGHT=650; PUBLIC_LINK=True
SAMPLING_FAMILIES=['top_right_identifier','long_glyph','large_dark_cluster1','pale_near_threshold','dark_pen']
SEMANTIC_LABELS=['ANNOTATION','PRINT','MIXED','UNCERTAIN','MASK_ERROR']
ANNOTATION_FAMILIES=['PALE_PENCIL','DARK_WRITING','PART_CATALOGUE','LONG_MARK_ARROW','OTHER']
print('Output:',RUN)

## 2. Resolve Notebook 59R3 outputs

In [ ]:
def require(c,m):
    if not c: raise RuntimeError(m)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def complete_runs(root):
    req={'retained_regions_clustered.csv','cluster_profiles.csv','special_family_summary.csv'}; hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and req.issubset({x.name for x in p.rglob('*.csv')}): hits.append(p)
    return sorted(set(hits),key=lambda p:p.stat().st_mtime)
def find_one(root,name):
    hits=[p for p in Path(root).rglob(name) if p.is_file()]
    require(len(hits)==1,f'Expected one {name} under {root}; found {len(hits)}.')
    return hits[0]
SOURCE=Path(SOURCE_59R3_RUN_OVERRIDE) if SOURCE_59R3_RUN_OVERRIDE else complete_runs(ROOT)[-1]
REGIONS_PATH=find_one(SOURCE / 'tables','retained_regions_clustered.csv')
CLUSTERS_PATH=find_one(SOURCE / 'tables','cluster_profiles.csv')
SPECIAL_PATH=find_one(SOURCE / 'tables','special_family_summary.csv')

print(SPECIAL_PATH)


# Notebook 64!
N64_SOURCE = ROOT / 'notebook64_exact_truth_and_threshold_safety'
pr=[p for p in N64_SOURCE.rglob('page_pair_resolution.csv') if p.is_file()]
if not pr: pr=[p for p in ROOT.rglob('page_pair_resolution.csv') if p.is_file()]
require(pr,'Could not locate page_pair_resolution.csv.')
PAGE_RESOLUTION_PATH=max(pr,key=lambda p:p.stat().st_mtime)
regions=pd.read_csv(REGIONS_PATH); resolution=pd.read_csv(PAGE_RESOLUTION_PATH)
require('page' in regions,'59R3 region table lacks page.')
require({'page','observed_path'}.issubset(resolution.columns),'page_pair_resolution.csv needs page and observed_path.')
regions['page']=regions.page.astype(str); resolution['page']=resolution.page.astype(str)
page_to_image=dict(zip(resolution.page,resolution.observed_path))
atomic_json(AUDIT/'resolved_sources.json',{'source_59r3_run':str(SOURCE),'regions_path':str(REGIONS_PATH),'regions_sha256':sha256_file(REGIONS_PATH),'page_resolution':str(PAGE_RESOLUTION_PATH)})
print('59R3:',SOURCE); print('Regions:',len(regions),'Pages:',regions.page.nunique())

## 3. Normalise fields and build machine-family flags

In [ ]:
def first_col(frame,names,required=True):
    for n in names:
        if n in frame.columns:
            return n
    if required:
        raise RuntimeError('Missing expected column; tried '+str(names))
    return None

COL={
    'region_id':first_col(regions,['region_id','retained_region_id','id']),
    'x':first_col(regions,['x','x0','bbox_x','left']),
    'y':first_col(regions,['y','y0','bbox_y','top']),
    'w':first_col(regions,['w','width','bbox_w']),
    'h':first_col(regions,['h','height','bbox_h']),
    'cluster':first_col(regions,['cluster','cluster_id','kmeans_cluster']),
    'area':first_col(regions,['area','pixels','region_pixels'],False),
    'median_p':first_col(regions,['median_probability','median_p_remove','median_p_remove_59r2','median_probability_59r2'],False),
    'mean_p':first_col(regions,['mean_probability','mean_p_remove','mean_p_remove_59r2'],False),
    'median_observed':first_col(regions,['median_observed_value','median_observed','mean_observed_value'],False),
    'aspect':first_col(regions,['aspect_ratio','bbox_aspect_ratio'],False),
    'context':first_col(regions,['context_density','local_context_density','mean_context_density','print_context_density'],False),
}

for k in ['x','y','w','h','cluster']:
    regions[COL[k]]=pd.to_numeric(regions[COL[k]],errors='raise')

def bool_series(names):
    c=first_col(regions,names,False)
    if c is None:
        return pd.Series(False,index=regions.index,dtype=bool),None
    s=regions[c]
    if pd.api.types.is_bool_dtype(s):
        out=s.fillna(False).astype(bool)
    elif pd.api.types.is_numeric_dtype(s):
        out=pd.to_numeric(s,errors='coerce').fillna(0).ne(0)
    else:
        out=s.fillna('').astype(str).str.strip().str.lower().isin(['true','1','yes','y','t'])
    return out.astype(bool),c

# Actual Notebook 59R3 output names are listed first. Historical aliases remain
# accepted so this reviewer can read either the executed 59R3 run or an older export.
top,ft=bool_series([
    'flag_top_right_identifier_candidate',
    'top_right_identifier_candidate','top_right_identifier','is_top_right_identifier',
])
longg,fl=bool_series([
    'flag_long_glyph_candidate',
    'long_glyph_candidate','long_glyph','is_long_glyph',
])
pale,fp=bool_series([
    'flag_pale_near_threshold',
    'pale_near_threshold','pale_near_threshold_candidate','is_pale_near_threshold',
])
dark,fd=bool_series([
    'flag_dark_pen_candidate',
    'dark_pen_candidate','dark_pen','is_dark_pen',
])
dense,fe=bool_series([
    'flag_dense_dark_candidate',
    'dense_dark_candidate','dense_dark','is_dense_dark',
])

regions['_flag_top_right']=top
regions['_flag_long_glyph']=longg
regions['_flag_large_dark_cluster1']=pd.to_numeric(regions[COL['cluster']],errors='coerce').eq(1)
regions['_flag_pale_near_threshold']=pale
regions['_flag_dark_pen']=dark
regions['_flag_dense_dark']=dense

flag_sources={
    'top_right_identifier':ft,
    'long_glyph':fl,
    'pale_near_threshold':fp,
    'dark_pen':fd,
    'dense_dark':fe,
}
availability={
    'top_right_identifier':int(top.sum()),
    'long_glyph':int(longg.sum()),
    'large_dark_cluster1':int(regions['_flag_large_dark_cluster1'].sum()),
    'pale_near_threshold':int(pale.sum()),
    'dark_pen':int(dark.sum()),
    'dense_dark':int(dense.sum()),
    'source_columns':flag_sources,
}
atomic_json(AUDIT/'sampling_family_availability.json',availability)

# Fail clearly rather than silently generating a one-family queue again.
missing_requested=[fam for fam in SAMPLING_FAMILIES if availability.get(fam,0)==0]
require(
    not missing_requested,
    'Requested sampling families have zero candidates: '
    +str(missing_requested)
    +'. Actual 59R3 columns include: '
    +str(sorted(c for c in regions.columns if c.startswith('flag_')))
)

print(json.dumps(availability,indent=2))


## 4. Build an initial page-diversified queue

In [ ]:
rng=np.random.default_rng(SEED)
used=set()
parts=[]
flags={
    'top_right_identifier':'_flag_top_right',
    'long_glyph':'_flag_long_glyph',
    'large_dark_cluster1':'_flag_large_dark_cluster1',
    'pale_near_threshold':'_flag_pale_near_threshold',
    'dark_pen':'_flag_dark_pen',
}

def diversified(frame,n):
    if frame.empty:
        return frame
    pages=list(frame.page.astype(str).unique())
    rng.shuffle(pages)
    buckets={p:frame[frame.page.astype(str)==p].index.to_list() for p in pages}
    for p in pages:
        rng.shuffle(buckets[p])
    chosen=[]
    while len(chosen)<min(n,len(frame)):
        progress=False
        for p in pages:
            if buckets[p] and len(chosen)<n:
                chosen.append(buckets[p].pop())
                progress=True
        if not progress:
            break
    return frame.loc[chosen]

# Families are sampled in the declared order. Once selected, a source region is not
# reused in another family, so the reviewer does not see duplicate cases.
for fam in SAMPLING_FAMILIES:
    flag_col=flags[fam]
    cand=regions[regions[flag_col] & ~regions.index.isin(used)].copy()
    take=diversified(cand,INITIAL_PER_FAMILY).copy()
    take['sampling_family']=fam
    parts.append(take)
    used.update(take.index)

nonempty=[p for p in parts if len(p)]
require(nonempty,'Review queue empty.')
queue=pd.concat(nonempty,axis=0).head(MAX_INITIAL_REVIEWS).copy()
queue['source_region_index']=queue.index.astype(int)
queue=queue.reset_index(drop=True)
queue['review_order']=np.arange(1,len(queue)+1)
queue['review_id']=[f'NB66V2-{i:04d}' for i in queue.review_order]
queue['semantic_label']=''
queue['annotation_family']=''
queue['reviewed_utc']=''

queue.to_csv(TABLES/'review_queue.csv',index=False)
coverage=(queue.groupby('sampling_family',as_index=False)
          .agg(regions=('review_id','size'),pages=('page','nunique')))
coverage.to_csv(TABLES/'review_queue_coverage.csv',index=False)

missing_queue_families=sorted(set(SAMPLING_FAMILIES)-set(coverage.sampling_family))
require(not missing_queue_families,'Queue omitted requested families: '+str(missing_queue_families))

print('Queue:',len(queue),'regions across',queue.page.nunique(),'pages')
display(coverage)


## 5. Durable review state and crop helper

In [ ]:
DECISIONS_PATH=REVIEW/'review_decisions.csv'; HISTORY_PATH=REVIEW/'review_history.jsonl'
def load_gray(page):
    p=Path(page_to_image[str(page)]); require(p.is_file(),f'Missing observed image: {p}'); im=cv2.imread(str(p),0); require(im is not None,f'Unreadable image: {p}'); return im
def crop_for_row(row,margin=CONTEXT_MARGIN):
    im=load_gray(row.page); x=int(row[COL['x']]); y=int(row[COL['y']]); w=int(row[COL['w']]); h=int(row[COL['h']]); H,W=im.shape; x0=max(0,x-margin); y0=max(0,y-margin); x1=min(W,x+w+margin); y1=min(H,y+h+margin); rgb=cv2.cvtColor(im[y0:y1,x0:x1],cv2.COLOR_GRAY2RGB); cv2.rectangle(rgb,(x-x0,y-y0),(min(rgb.shape[1]-1,x+w-x0),min(rgb.shape[0]-1,y+h-y0)),(220,40,40),2); return rgb
def persist(event=None):
    queue.to_csv(DECISIONS_PATH,index=False)
    if event:
        with open(HISTORY_PATH,'a') as f:f.write(json.dumps(event,default=str)+'\n')
def first_unreviewed():
    q=queue.index[queue.semantic_label.astype(str).str.len()==0].tolist(); return q[0] if q else 0
persist(); print('Review state:',DECISIONS_PATH)

## 6. Minimal two-label reviewer

In [ ]:
def machine_info(row):
    d={'sampling_family':row.sampling_family,'cluster':row[COL['cluster']],'top_right_flag':bool(row._flag_top_right),'long_glyph_flag':bool(row._flag_long_glyph),'large_dark_cluster1':bool(row._flag_large_dark_cluster1),'pale_near_threshold_flag':bool(row._flag_pale_near_threshold),'high_context_overlap_flag':bool(row._flag_high_context_overlap),'dark_pen_flag':bool(row._flag_dark_pen)}
    for label,key in [('area','area'),('median_p','median_p'),('mean_p','mean_p'),('median_observed','median_observed'),('aspect','aspect')]:
        c=COL.get(key)
        if c is not None:d[label]=row[c]
    return json.dumps(d,indent=2,default=str)
def render(i):
    i=int(max(0,min(int(i),len(queue)-1))); row=queue.iloc[i]; done=int((queue.semantic_label.astype(str).str.len()>0).sum()); title=f'Review {i+1}/{len(queue)} | completed {done}/{len(queue)} | page {row.page}'; sem=row.semantic_label or None; fam=(row.annotation_family if row.annotation_family and row.annotation_family!='NOT_APPLICABLE' else None); mach=machine_info(row) if sem else 'Machine diagnostics hidden until this item is saved.'; return crop_for_row(row),title,sem,fam,mach,i
def save_next(i,semantic,family):
    i=int(i)
    if semantic not in SEMANTIC_LABELS:raise gr.Error('Choose label 1.')
    if semantic in ['ANNOTATION','MIXED']:
        if family not in ANNOTATION_FAMILIES:raise gr.Error('Choose label 2.')
    else:family='NOT_APPLICABLE'
    now=datetime.now(timezone.utc).isoformat(); queue.at[i,'semantic_label']=semantic; queue.at[i,'annotation_family']=family; queue.at[i,'reviewed_utc']=now; persist({'review_id':queue.at[i,'review_id'],'semantic_label':semantic,'annotation_family':family,'utc':now})
    later=queue.index[(queue.index>i)&(queue.semantic_label.astype(str).str.len()==0)].tolist(); remaining=queue.index[queue.semantic_label.astype(str).str.len()==0].tolist(); ni=later[0] if later else (remaining[0] if remaining else i); image,title,sem,fam2,mach,idx=render(ni); return image,title,sem,fam2,mach,idx,f'Saved {queue.at[i,"review_id"]}: {semantic} / {family}'
def previous(i):return render(max(0,int(i)-1))
def next_item(i):return render(min(len(queue)-1,int(i)+1))
with gr.Blocks(title='Notebook 66 v2.1') as app:
    gr.Markdown('# Notebook 66 v2.1 - Minimal Semantic Review\nMake the visual judgement first. Machine evidence is hidden until the item is first saved.')
    idx=gr.State(first_unreviewed()); title=gr.Textbox(label='Progress',interactive=False); image=gr.Image(label='Observed context - red box is the 59R3 region',height=DISPLAY_HEIGHT); semantic=gr.Radio(SEMANTIC_LABELS,label='1. What is this region?'); family=gr.Radio(ANNOTATION_FAMILIES,label='2. If annotation/mixed, what kind?');
    with gr.Row(): prev=gr.Button('Previous'); save=gr.Button('Save + Next',variant='primary'); nxt=gr.Button('Next')
    status=gr.Textbox(label='Save status',interactive=False); machine=gr.Code(label='Machine evidence (shown after save)',language='json')
    app.load(render,[idx],[image,title,semantic,family,machine,idx]); prev.click(previous,[idx],[image,title,semantic,family,machine,idx]); nxt.click(next_item,[idx],[image,title,semantic,family,machine,idx]); save.click(save_next,[idx,semantic,family],[image,title,semantic,family,machine,idx,status])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 7. Summarise the human labels

In [ ]:
reviewed=pd.read_csv(DECISIONS_PATH,dtype=str).fillna(''); completed=reviewed[reviewed.semantic_label!=''].copy(); completed['annotation_or_mixed']=completed.semantic_label.isin(['ANNOTATION','MIXED']); completed['part_catalogue']=completed.annotation_family.eq('PART_CATALOGUE'); completed['long_mark_arrow']=completed.annotation_family.eq('LONG_MARK_ARROW'); completed['dark_writing']=completed.annotation_family.eq('DARK_WRITING'); completed['pale_pencil']=completed.annotation_family.eq('PALE_PENCIL')
if len(completed):
    summary=completed.groupby('sampling_family',as_index=False).agg(reviewed=('review_id','size'),pages=('page','nunique'),annotation_or_mixed=('annotation_or_mixed','sum'),part_catalogue=('part_catalogue','sum'),long_mark_arrow=('long_mark_arrow','sum'),dark_writing=('dark_writing','sum'),pale_pencil=('pale_pencil','sum'),print_regions=('semantic_label',lambda s:int((s=='PRINT').sum())),uncertain_regions=('semantic_label',lambda s:int((s=='UNCERTAIN').sum())),mask_error_regions=('semantic_label',lambda s:int((s=='MASK_ERROR').sum()))); summary['annotation_or_mixed_purity']=summary.annotation_or_mixed/summary.reviewed
else:summary=pd.DataFrame()
summary.to_csv(TABLES/'human_review_summary_by_family.csv',index=False); completed.to_csv(TABLES/'human_review_completed.csv',index=False)
progress={'queue_size':len(reviewed),'completed':len(completed),'remaining':len(reviewed)-len(completed),'completion_fraction':len(completed)/max(len(reviewed),1),'pages_reviewed':int(completed.page.nunique()) if len(completed) else 0}; atomic_json(RUN/'review_progress.json',progress); print(json.dumps(progress,indent=2)); display(summary)

## 8. Curated upload package

In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'gradio':gr.__version__,'platform':platform.platform()}; identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'Notebook66-v2.1-minimal-human-semantic-review','source_59r3_run':str(SOURCE),'source_regions_sha256':sha256_file(REGIONS_PATH),'required_human_labels':['semantic_label','annotation_family'],'initial_per_family':INITIAL_PER_FAMILY,'initial_target_reviews':len(queue),'machine_evidence_hidden_before_first_save':True,'promotion_authority':False,'training_truth_status':'review evidence only unless separately promoted under project governance','software':environment}; atomic_json(RUN/'experiment_identity.json',identity)
reviewed=pd.read_csv(DECISIONS_PATH,dtype=str).fillna(''); completed=reviewed[reviewed.semantic_label!='']; lines=['NOTEBOOK 66 V2.1 - MINIMAL HUMAN SEMANTIC REVIEW','',f'Source 59R3: {SOURCE}',f'Queue: {len(reviewed)}',f'Completed: {len(completed)}',f'Remaining: {len(reviewed)-len(completed)}','','Required label 1: ANNOTATION / PRINT / MIXED / UNCERTAIN / MASK_ERROR','Required label 2 for annotation/mixed: PALE_PENCIL / DARK_WRITING / PART_CATALOGUE / LONG_MARK_ARROW / OTHER','','Evidence collection only. No automatic training or promotion authority.']; (RUN/'executive_summary.txt').write_text('\n'.join(lines)+'\n')
curated=[RUN/'executive_summary.txt',RUN/'experiment_identity.json',RUN/'review_progress.json',TABLES/'review_queue.csv',TABLES/'review_queue_coverage.csv',REVIEW/'review_decisions.csv',AUDIT/'resolved_sources.json',AUDIT/'sampling_family_availability.json']
for p in [TABLES/'human_review_summary_by_family.csv',TABLES/'human_review_completed.csv']:
    if p.is_file():curated.append(p)
manifest=[]
for src in curated:
    require(src.is_file(),f'Missing {src}'); rel=src.relative_to(RUN); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); require(sha256_file(src)==sha256_file(dst),f'Hash mismatch {rel}'); manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).drop_duplicates('relative_path').sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 66 v2 minimal human semantic review package.\nOnly two human labels are required per region.\nNo automatic training, promotion or production authority.\n')
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+'\n')
zip_path=RUN/'Notebook66_v2_Minimal_Human_Semantic_Review_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file():zf.write(p,p.relative_to(UPLOAD))
require(zip_path.is_file() and zip_path.stat().st_size>0,'ZIP failed.'); print('\n'.join(lines)); print('Upload ZIP:',zip_path)